In [1]:
# %pip install

In [45]:
import requests
import time
import pandas as pd

#### FETCH LEGITIMATE URLS DATA

In [5]:
def fetch_from_coingecko(url):
    page = 1
    all_exchanges = []

    while True:
        # Add the page parameter to the request
        params = {"per_page": 300, "page": page}

        try:
            # Make the request
            response = requests.get(url, params=params)

            # Check if the request was successful
            if response.status_code == 200:
                data = response.json()
                # If no data is returned, stop the loop
                if not data:
                    break
                # Add the fetched data to the list
                all_exchanges.extend(data)
                print(f"Fetched page {page} with {len(data)} exchanges.")
                # Move to the next page
                page += 1
                # Add a delay to avoid hitting the rate limit
                time.sleep(1)

            elif response.status_code == 429:
                # Handle rate limit error
                print("Rate limit exceeded. Waiting for 60 seconds before retrying...")
                time.sleep(60)  # Wait 60 seconds before retrying

            else:
                # Handle other errors
                print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
                break

        except requests.exceptions.RequestException as e:
            # Handle network errors
            print(f"Network error: {e}")
            break

    return all_exchanges

# Function to fetch market data (coins list)
def fetch_coin_market_data(url):
    page = 1
    all_coins = []


    while True:
      params = {'vs_currency': 'usd', 'order': 'market_cap_desc', 'per_page': 250, 'page': page, 'x_cg_demo_api_key': "CG-Y7Gt7HP5cmaK74WbKKbhh7dZ"}

      try:
          response = requests.get(url, params=params)

          if response.status_code == 200:
                data = response.json()
                # If no data is returned, stop the loop
                if not data:
                    break
                # Add the fetched data to the list
                all_coins.extend(data)
                print(f"Fetched page {page} with {len(data)} exchanges.")
                # Move to the next page
                page += 1
                # Add a delay to avoid hitting the rate limit
                time.sleep(1)

          elif response.status_code == 429:
                # Handle rate limit error
                print("Rate limit exceeded. Waiting for 60 seconds before retrying...")
                time.sleep(60)  # Wait 60 seconds before retrying

          else:
                # Handle other errors
                print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
                break

      except requests.exceptions.RequestException as e:
            # Handle network errors
            print(f"Network error: {e}")
            break

    return all_coins


# Function to fetch coin details by coin ID
def fetch_coin_details(coin_id):
    while True:
          details_url = f'https://api.coingecko.com/api/v3/coins/{coin_id}'
          details_response = requests.get(details_url, headers={'x_cg_demo_api_key': "CG-Y7Gt7HP5cmaK74WbKKbhh7dZ"})

          if details_response.status_code == 200:
              coin_details = details_response.json()
              homepage_list = coin_details.get('links', {}).get('homepage', [])
              homepage_url = homepage_list[0] if homepage_list else 'N/A'
              print(f"Fetched details for coin {coin_id}: {homepage_url}")
              # Add a delay to avoid hitting the rate limit
              time.sleep(1)

              return homepage_url

          elif details_response.status_code == 429:
              print("Rate limit exceeded, waiting for 60 seconds before retrying...")
              time.sleep(60)

          else:
              print(f"Failed to fetch details for coin {coin_id}. Status code: {details_response.status_code}")
              return 'N/A'

# Function to fetch exchanges from DeFi Llama
def fetch_from_other_source(url):
    # Make the request
    response = requests.get(url)
    if response.status_code == 200:
        # Return the JSON data
        return response.json()
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")

        return []


In [6]:
# function to convert to dictionary
def convert_to_dict(items):
    # Create a list to store the name-url mappings
    dict_list = []

    # Process each URL in the list
    for url in items:
        # Extract the text before the domain extension
        name = url.split('.')[0]
        # Create a dictionary for the name and URL
        dict_list.append({'name': name, 'url': url})

    return dict_list

# function to get the format a domain to URL
def format_domain(domain):
    if not domain.startswith(('https://', 'http://', 'www.')):
        return 'https://' + domain
    return domain

In [7]:
crypto_data_urls = []

In [8]:
# fetch centralized exchanges from coingecko
cex_url = "https://api.coingecko.com/api/v3/exchanges"
gecko_data = fetch_from_coingecko(cex_url)

for exchange in gecko_data:
    # Extract the name from the exchange data
    url = exchange['url']

    # Append the name to the list
    crypto_data_urls.append(url)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(gecko_data)}, {gecko_data[0]}")

Fetched page 1 with 300 exchanges.
Fetched page 2 with 300 exchanges.
Fetched page 3 with 300 exchanges.
Fetched page 4 with 48 exchanges.
Total exchanges fetched: 948, {'id': 'binance', 'name': 'Binance', 'year_established': 2017, 'country': 'Cayman Islands', 'description': 'One of the world’s largest cryptocurrency exchanges by trading volume, offering a wide range of services including spot, futures, and staking options.', 'url': 'https://www.binance.com/', 'image': 'https://coin-images.coingecko.com/markets/images/52/small/binance.jpg?1706864274', 'has_trading_incentive': False, 'trust_score': 10, 'trust_score_rank': 1, 'trade_volume_24h_btc': 336258.1016595171, 'trade_volume_24h_btc_normalized': 220247.5282068443}


In [ ]:
# fetch centralized exchanges from coingecko
gecko_coin_url = 'https://api.coingecko.com/api/v3/coins/markets'
gecko_coin_data = fetch_coin_market_data(gecko_coin_url)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(gecko_coin_data)}, {gecko_coin_data[0]}")

In [ ]:
for exchange in gecko_coin_data:
    # Extract the name from the exchange data
    url = fetch_coin_details(exchange["id"])
    # print(url)
    # Append the name to the list
    crypto_data_urls.append(url)

In [16]:
pap_exchanges = fetch_from_other_source("https://api.coinpaprika.com/v1/exchanges")

for exchange in pap_exchanges:
    if exchange['links'] and exchange['links'].get('website'):
        url = exchange['links']['website'][0]
    else:
        url = None

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(pap_exchanges)}, {pap_exchanges[0]}")

Total exchanges fetched: 1026, {'id': 'defi-scan', 'name': 'DeFi Scan', 'description': '', 'active': True, 'website_status': True, 'api_status': False, 'message': 'No current data', 'links': {'twitter': ['https://twitter.com/defichain'], 'website': ['https://defiscan.live/']}, 'markets_data_fetched': False, 'adjusted_rank': None, 'reported_rank': None, 'currencies': 1, 'markets': 0, 'fiats': [], 'quotes': {'USD': {'reported_volume_24h': 0, 'adjusted_volume_24h': 0, 'reported_volume_7d': 0, 'adjusted_volume_7d': 0, 'reported_volume_30d': 0, 'adjusted_volume_30d': 0}}, 'last_updated': '2025-04-13T23:19:24Z', 'sessions_per_month': 83865, 'confidence_score': 0}


In [ ]:
llama_url = "https://api.llama.fi/protocols"
llama_data = fetch_from_other_source(llama_url)

for exchange in llama_data:
    # Extract the name from the exchange data
    url = exchange['url']
    
    # Append the name to the list
    crypto_data_urls.append(url)

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(llama_data)}, {llama_data[0]}")

Total exchanges fetched: 5768, {'id': '2269', 'name': 'Binance CEX', 'address': None, 'symbol': '-', 'url': 'https://www.binance.com', 'description': 'Binance is a cryptocurrency exchange which is the largest exchange in the world in terms of daily trading volume of cryptocurrencies', 'chain': 'Multi-Chain', 'logo': 'https://icons.llama.fi/binance-cex.jpg', 'audits': '0', 'audit_note': None, 'gecko_id': None, 'cmcId': None, 'category': 'CEX', 'chains': ['Bitcoin', 'Ethereum', 'Binance', 'Solana', 'Tron', 'Ripple', 'Doge', 'Base', 'Arbitrum', 'Avalanche', 'Litecoin', 'Optimism', 'Polkadot', 'Hedera', 'TON', 'Near', 'Stellar', 'Algorand', 'Polygon', 'Aptos', 'Starknet', 'Chiliz', 'Manta', 'Celo', 'Op_Bnb', 'Ronin', 'Scroll', 'Sui', 'zkSync Era'], 'module': 'binance/index.js', 'twitter': 'binance', 'forkedFrom': [], 'oracles': [], 'listedAt': 1668170565, 'methodology': 'We collect the wallets from this binance blog post https://www.binance.com/en/blog/community/our-commitment-to-transpare

In [23]:
crypto_data_urls = [item for sublist in crypto_data_urls for item in (sublist if isinstance(sublist, list) else [sublist])]
crypto_data_urls = [url for url in crypto_data_urls if url != '' and url != 'N/A']
crypto_data_urls = set(crypto_data_urls)
crypto_data_urls = list(crypto_data_urls)

In [26]:
print(f"Total crypto url fetched: {len(crypto_data_urls)}")

Total crypto url fetched: 18596


In [ ]:
# create a DataFrame from the list of URLs
crypto_urls_df = pd.DataFrame(crypto_data_urls, columns=['url'])
# save the DataFrame to a CSV file
crypto_urls_df.to_csv('crypto_urls.csv', index=False)

In [ ]:
# fetch other legitimate websites asides crypto
# other_websites = "https://raw.githubusercontent.com/seigdev/resources/refs/heads/main/top-websites.csv"

# other_raw = fetch_data_csv(other_websites)

# others_df = pd.DataFrame(other_raw)

# others_df = others_df.drop(columns=0)

# others_df = others_df.rename(columns={1: 'url'})

In [29]:
df_raw = pd.read_csv("crypto_urls.csv")

legit_df = df_raw[['url']].copy()

# assign string labels to legitimate urls
legit_df.loc[:, 'label'] = 'legit'

# assign string labels to legitimate urls
legit_df.loc[:, 'label_no'] = 0

legit_df

,url,label,label_no
0,https://unixgaming.org,legit,0
1,https://app.bakedpotatoes.dog,legit,0
2,https://realtrumpmania.com/,legit,0
3,https://behodler.io,legit,0
4,https://merchdao.com/,legit,0
...,...,...,...
18591,https://simonnycmayor.com/,legit,0
18592,https://padre.gg,legit,0
18593,https://www.bubblu.xyz/,legit,0
18594,https://illuminex.xyz,legit,0


#### FETCH SCAM URLS DATA

In [ ]:
# url to fetch scam urls from eth-phishing-detect
scam_url = "https://raw.githubusercontent.com/MetaMask/eth-phishing-detect/master/src/config.json"
scam_ex = fetch_from_other_source(scam_url)

# fetch the list of blacklist urls
blacklist = scam_ex["blacklist"]

In [39]:
blacklist_raw = pd.DataFrame(blacklist, columns=['url'])

# copy the name and urls columns to a new dataframe
scam_df = blacklist_raw[['url']].copy()

scam_df["url"] = scam_df["url"].apply(format_domain)

# assign string labels to legitimate urls
scam_df.loc[:, 'label'] = 'scam'

# assign string labels to legitimate urls
scam_df.loc[:, 'label_no'] = 1

scam_df = scam_df.sample(n=50000, random_state=42).reset_index(drop=True)

scam_df

,url,label,label_no
0,https://nagapaynft.pages.dev,scam,1
1,https://jito-jto.com,scam,1
2,https://mocaversenetworkfi.com,scam,1
3,https://ocean-bounty.net,scam,1
4,https://dydxtrade.support,scam,1
...,...,...,...
49995,https://metaskmak-chrome.gitbook.io,scam,1
49996,https://connect-launch.network,scam,1
49997,https://openseaproj11.vercel.app,scam,1
49998,https://yoracletradingdefi.app,scam,1


In [40]:
seed = 42

In [42]:
# merge both the legit and scam urls together
all_urls_df = pd.concat([legit_df, scam_df], ignore_index=True)

# shuffle the urls across the dataframe
all_urls_df = all_urls_df.sample(frac=1, random_state=seed).reset_index(drop=True)

all_urls_df

,url,label,label_no
0,https://proposal-availfoundation.com,scam,1
1,https://wesp.cloud,scam,1
2,https://longevity-vibe.com,scam,1
3,https://stakleand.com,scam,1
4,https://kavacheck.xyz,scam,1
...,...,...,...
68591,https://scaliatokendefi.app,scam,1
68592,http://bitmark.co,legit,0
68593,https://xrpevent-prize.com,scam,1
68594,https://metf.finance/#/,legit,0


In [ ]:
all_urls_df.to_json('crypto_url_data.json', orient='records', lines=False)